## 7. Group Assignment & Presentation


__You should be able to start up on this exercise after Lecture 1.__

*This exercise must be a group effort. That means everyone must participate in the assignment.*

In this assignment you will solve a data science problem end-to-end, pretending to be recently hired data scientists in a company. To help you get started, we've prepared a checklist to guide you through the project. Here are the main steps that you will go through:

4. Prepare the data to better expose the underlying data patterns to machine learning algorithms
5. Explore many different models and short-list the best ones
6. Fine-tune your models
7. Present your solution (video presentation) 

In each step we list a set of questions that one should have in mind when undertaking a data science project. The list is not meant to be exhaustive, but does contain a selection of the most important questions to ask. We will be available to provide assistance with each of the steps, and will allocate some part of each lesson towards working on the projects.

Your group must submit a _**single**_ Jupyter notebook, structured in terms of the first 6 sections listed above (the seventh will be a video uploaded to some streaming platform, e.g. YouTube, Vimeo, etc.).

### 3. Explore the data
1. Create a copy of the data for explorations (sampling it down to a manageable size if necessary)
2. Create a Jupyter notebook to keep a record of your data exploration
3. Study each feature and its characteristics:
    * Name
    * Type (categorical, int/float, bounded/unbounded, text, structured, etc)
    * Percentage of missing values
    * Check for outliers, rounding errors etc
4. For supervised learning tasks, identify the target(s)
5. Visualise the data
6. Study the correlations between features
7. Identify the promising transformations you may want to apply (e.g. convert skewed targets to normal via a log transformation)
8. Document what you have learned
### 4. Prepare the data
Notes:
* Work on copies of the data (keep the original dataset intact).
* Write functions for all data transformations you apply, for three reasons:
    * So you can easily prepare the data the next time you run your code
    * So you can apply these transformations in future projects
    * To clean and prepare the test set
    
    
1. Data cleaning:
    * Fix or remove outliers (or keep them)
    * Fill in missing values (e.g. with zero, mean, median, regression ...) or drop their rows (or columns)
2. Feature selection (optional):
    * Drop the features that provide no useful information for the task (e.g. a customer ID is usually useless for modelling).
3. Feature engineering, where appropriate:
    * Discretize continuous features
    * Use one-hot encoding if/when relevant
    * Add promising transformations of features (e.g. $\log(x)$, $\sqrt{x}$, $x^2$, etc)
    * Aggregate features into promising new features
4. Feature scaling: standardise or normalise features
### 5. Short-list promising models
We expect you to do some additional research and train at **least one model per team member**.

1. Train mainly quick and dirty models from different categories (e.g. linear, SVM, Random Forests etc) using default parameters
2. Measure and compare their performance
3. Analyse the most significant variables for each algorithm
4. Analyse the types of errors the models make
5. Have a quick round of feature selection and engineering if necessary
6. Have one or two more quick iterations of the five previous steps
7. Short-list the top three to five most promising models, preferring models that make different types of errors
### 6. Fine-tune the system
1. Fine-tune the hyperparameters
2. Once you are confident about your final model, measure its performance on the test set to estimate the generalisation error
### 7. Present your solution
1. Document what you have done
2. Create a nice 15 minute video presentation with slides
    * Make sure you highlight the big picture first
3. Explain why your solution achieves the business objective
4. Don't forget to present interesting points you noticed along the way:
    * Describe what worked and what did not
    * List your assumptions and you model's limitations
5. Ensure your key findings are communicated through nice visualisations or easy-to-remember statements (e.g. "the median income is the number-one predictor of housing prices")
6. Upload the presentation to some online platform, e.g. YouTube or Vimeo, and supply a link to the video in the notebook.
Géron, A. 2017, *Hands-On Machine Learning with Scikit-Learn and Tensorflow*, Appendix B, O'Reilly Media, Inc., Sebastopol.

# Research, reasons for no show health care appointments.

## 1. Framing the Problem and looking at the bigger picture.

### Objective:
Predict and analyze healthcare appointment no-shows, focusing on economic and demographic factors that might influence them.

#### Key Questions with Demographics:
- What economic factors (e.g., income, unemployment rate, access to transportation) correlate with no-shows?
- Are there specific patterns in different Brazilian cities or regions?
- Do specific age groups (like the elderly or youth) show higher no-show rates?
- How does education or literacy correlate with appointment attendance?
- Are there patterns in no-shows based on gender or ethnicity in different cities?
- Is family structure (e.g., single parents vs. larger households) a factor?

#### Scope Expansion:
- Geographical Focus: Cities in Brazil (urban/rural divide? Large vs. small cities?).
- Economic Focus: Indicators like GDP per capita, public health investment, and local employment rates.
- Healthcare Metrics: Number of appointments, no-show rates, reasons for no-shows.
- Income brackets.
- Local employment levels.
- Urbanization metrics.

#### Feature Engineering for Demographics:
- Combine Age (from the healthcare dataset) with IDHM_Renda or GDP_CAPITA (from the Brazil cities dataset) to explore how income levels vary by age group.
- Create a composite variable for Neighbourhood-level healthcare data and RURAL_URBAN classification to highlight urban-rural divides.
- Group cities by demographic trends (e.g., predominantly elderly populations).
- Group cities based on IDHM scores to categorize them into high, medium, and low human development zones.

#### Insights from Demographic-Economic Analysis:
- Identify high no-show rates in low-income areas by linking Neighbourhood data with GDP_CAPITA or IDHM_Renda.
- Highlight rural cities (RURAL_URBAN = "Rural") with low MUN_EXPENDIT for targeted healthcare funding.

## 2. Get the Data  
The healthcare no-show data was sourced from Kaggle, providing a clean and well-organized dataset. After a quick exploration of the dataset, we thought it would be interesting to investigate potential correlations with Brazil's economic and demographic aspects. To support this, we located another dataset containing information about Brazilian cities, though it requires significant cleaning and preprocessing due to its raw state.

## 3. Explore and prepare the data

#### 3.1 Imports

In [19]:
# !pip install plotly
# !pip install nbformat
# !pip install geopandas contextily
# !pip install folium
# !pip install shapely
# !pip install osmnx
# !pip install seaborn

import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import osmnx as ox
from sklearn.preprocessing import OneHotEncoder
import folium
from shapely.geometry import Polygon, mapping
import geopandas as gpd
import json
import unicodedata

#### 3.2 Data set loading

In [20]:
df_appointments = pd.read_csv('healthcare-noshow/healthcare_noshows_appt.csv', thousands=',')
df_neighbourhoods = pd.read_csv('neighbourhoods/Vitoria_Economic_Data.csv', thousands=',')
with open('neighbourhoods/vitoria_neighbourhoods_geodata.json', 'r', encoding='utf-8') as f:
    geodata_neighbourhoods = json.load(f)
    # Minor cleaning steps:
df_neighbourhoods_cleaned = df_neighbourhoods.copy()
df_appointments_cleaned = df_appointments.copy()

columns_to_clean = ['Population aged 0 to 4']

df_neighbourhoods_cleaned[columns_to_clean] = df_neighbourhoods[columns_to_clean].replace('-', 0).apply(pd.to_numeric, errors='coerce')

#### 3.3 Study the features and characteristics

In [ ]:
# Features study
def study_features(df):
    feature_types: dict[str, str] = {}
    feature_summary = []
    for column in df.columns:
        col_type = df[column].dtype
        
        # Check for every possible number type:
        if np.issubdtype(col_type, np.number):
            feature_type = "Numerical"
        elif col_type == "object":
            if len(df[column].unique()) == 2:
                feature_type = "Binary"
            else:
                feature_type = "Categorical"
        else:
            if len(df[column].unique()) == 2:
                feature_type = "Binary"
            else:
                feature_type = "Other"

        # Feature analysis
        feature_info = {
            "Feature Name": column,
            "Type": feature_type,
            "Missing Values (%)": df[column].isnull().mean() * 100,
            "Unique Values": len(df[column].unique()),
        }

        # Add numeric details if applicable
        if feature_type == "Numerical":
            feature_info.update({
                "Min": df[column].min(),
                "Max": df[column].max(),
                "Mean": df[column].mean(),
                "Std Dev": df[column].std(),
            })

            # Check for outliers using IQR
            q1 = df[column].quantile(0.25)
            q3 = df[column].quantile(0.75)
            iqr = q3 - q1
            outliers = df[(df[column] < (q1 - 1.5 * iqr)) | (df[column] > (q3 + 1.5 * iqr))]
            feature_info["Outliers (Count)"] = len(outliers)

        feature_summary.append(feature_info)
        feature_types[column] = feature_type

    return pd.DataFrame(feature_summary), feature_types

# Analyze the datasets
healthcare_feature_summary, healthcare_feature_type = study_features(df_appointments_cleaned)
display(healthcare_feature_summary)

neighbourhoods_feature_summary, neighbourhoods_feature_type = study_features(df_neighbourhoods_cleaned)
display(neighbourhoods_feature_summary)

### 3.4 Explore map data
Let's also take a look at the neighbourhood boundaries we extracted

In [ ]:
# Extract and convert ESRI geometry to GeoDataFrame
polygons = []
names = []

for feature in geodata_neighbourhoods["features"]:
    rings = feature["geometry"]["rings"][0]  # Assuming single ring for simplicity
    polygons.append(Polygon(rings))
    names.append(feature["attributes"]["nome"])

gdf = gpd.GeoDataFrame({"nome": names, "geometry": polygons})

# Initialize a folium map centered on the data
m = folium.Map(location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10)

# Add polygons to the map
for _, row in gdf.iterrows():
    geo_json = mapping(row.geometry)  # Convert geometry to GeoJSON-like dict
    folium.GeoJson(
        geo_json,
        name=row['nome'],
        tooltip=folium.Tooltip(row['nome']),  # Add tooltips with names
        style_function=lambda x: {
            "fillColor": "red",
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.6
        },
    ).add_to(m)

# Display the map
m.save("neighbourhood_map.html")
m

### 3.5 Mapping categorical data / Data encoding
We have decided to encode the data into simpler values.
If the feature type is Binary we will assume they are boolean and try to cast them. If that fails we will apply One-Hot Encoding.
If the feature is not Binary, we will apply a frequency encoding on the data.

We refer to Binary as a column that has only 2 Unique values.

In [ ]:
def encode_data(df, feature_types, ignore_columns=[]):
    """
    Perform one-hot encoding on a DataFrame based if the feature is binary.
    """
    encoded_df = df.copy()
    for column, feature_type in feature_types.items():
        if column not in df.columns or column in ignore_columns:
            continue
        
        if feature_type == "Binary":
            try:
                encoded_df[column] = encoded_df[column].astype(int)
            except ValueError:
                print(f'Column is not boolean: {column}. Will try to apply One-Hot Encoding.')
                val1, val2 = df[column].unique()
                # Create one-hot encoded columns
                encoded_df[f"{column}_{val1}"] = (df[column] == val1).astype(int).apply(pd.to_numeric, errors='coerce')
                encoded_df[f"{column}_{val2}"] = (df[column] == val2).astype(int).apply(pd.to_numeric, errors='coerce')
                
                # Drop the original column
                encoded_df.drop(columns=[column], inplace=True)
        
        # Frequency encoding for categorical features
        elif feature_type == "Categorical":
            frequency_map = df[column].value_counts(normalize=True)
                    
            # Replace categories with their frequencies
            encoded_df[f"{column}_freq_encoded"] = df[column].map(frequency_map).apply(pd.to_numeric, errors='coerce')
            
            # Drop the original categorical column
            encoded_df.drop(columns=[column], inplace=True)
            # Numerical columns remain unchanged
        elif feature_type == "Numerical":
            encoded_df[column] = df[column].apply(pd.to_numeric, errors='coerce')
            continue
        else:
            print(f"Warning: Unsupported feature type '{feature_type}' for column '{column}'. Skipping.")

    return encoded_df
    
df_appointments_encoded = encode_data(df_appointments_cleaned, healthcare_feature_type, ["Gender", "Neighbourhood"])
df_appointments_encoded["Gender"] = df_appointments_encoded['Gender'].map({'F': 0, 'M': 1})
display(df_appointments_encoded)
df_neighbourhoods_encoded = encode_data(df_neighbourhoods_cleaned, neighbourhoods_feature_type, ["Neighborhood name"])
display(df_neighbourhoods_encoded)

### 3.6 Correlation matrixes for our data

First we need to merge the data. Let's see how many names of the neighbourhoods match intersect between the two datasets

In [ ]:
intersection = set(df_appointments_encoded['Neighbourhood'].unique()).intersection(set(df_neighbourhoods_encoded['Neighborhood name'].unique()))
display(intersection)

As we can see there is no intersection between the two columns with the accents.. 

Let's remove the accents, remove punctuation and spaces at the end.

In [ ]:
df_neighbourhoods_encoded['Neighborhood name'] = df_neighbourhoods_encoded['Neighborhood name'].str.upper()
df_appointments_encoded['Neighbourhood'] = df_appointments_encoded['Neighbourhood'].str.upper()

def remove_accents(input_str):
    if input_str.endswith(' '):
        input_str = input_str[:-1]
    if "'" in input_str:
        input_str = input_str.replace("'", " ")
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    return ''.join([c for c in nfkd_form if not unicodedata.combining(c)])


df_neighbourhoods_encoded['Neighborhood name'] = df_neighbourhoods_encoded['Neighborhood name'].apply(remove_accents)
df_appointments_encoded['Neighbourhood'] = df_appointments_encoded['Neighbourhood'].apply(remove_accents)

intersection = set(df_appointments_encoded['Neighbourhood'].unique()).intersection(set(df_neighbourhoods_encoded['Neighborhood name'].unique()))
display(intersection)
display(len(intersection))

# Print out different neighbourhood names (unique)
neighborhoods_appointments_diff = df_appointments_encoded['Neighbourhood'].str.strip().str.lower()
neighborhoods_neighbourhoods_diff = df_neighbourhoods_encoded['Neighborhood name'].str.strip().str.lower()

set_appointments = set(neighborhoods_appointments_diff)
set_neighbourhoods = set(neighborhoods_neighbourhoods_diff)

unique_to_appointments = set_appointments - set_neighbourhoods
unique_to_neighbourhoods = set_neighbourhoods - set_appointments

print("Neighborhoods unique to df_appointments_cleaned:")
for name in unique_to_appointments:
    print(name)
print(f"\nNumber of neighborhoods unique to df_appointments_cleaned: {len(unique_to_appointments)}")
print("\nNeighborhoods unique to df_neighbourhoods_cleaned:")
for name in unique_to_neighbourhoods:
    print(name)
print(f"\nNumber of neighborhoods unique to df_neighbourhoods_cleaned: {len(unique_to_neighbourhoods)}")

As we can see now we match with 78 out of 81 neighbourhoods. That is a lot better, but we still have one instance where one dataset has 'comdusa' neighbourhood, and another one has 'condusa'. As it was decided that it was a misspell we change that as well.

In [26]:
df_appointments_encoded.rename(columns={'condusa': 'comdusa'}, inplace=True)

Let's now merge the two datasets and observer the correlation matrix.

In [ ]:
def plot_correlation_matrix(matrix, title, figsize=(10, 10), annot=True):
    plt.figure(figsize=figsize)
    sns.heatmap(matrix, annot=annot, cmap='coolwarm', center=0)
    plt.title(title)
    plt.show()

df_merged = pd.merge(df_appointments_encoded,
                    df_neighbourhoods_encoded,
                    left_on="Neighbourhood",
                    right_on="Neighborhood name",
                    how="left")

# Drop the duplicate columns or columns that are not needed
df_merged.drop(columns=["Neighborhood name"], inplace=True)
df_merged.drop(columns=["PatientId"], inplace=True)
df_merged.drop(columns=["AppointmentID"], inplace=True)

# Make a copy of the merged DataFrame
df_copy = df_merged.copy()

# Drop the columns that are not needed for the correlation matrix
df_copy.drop(columns=["Neighbourhood"], inplace=True)

# Now we can create a correlation matrix
correlation_matrix = df_copy.corr()
plot_correlation_matrix(correlation_matrix, "Correlation Matrix", figsize=(20, 20), annot=False)

Based on the correlation martix above we can see that there is not a lot of correlation between any of the features we have with the "Showed_up" feature. We will like to get a closer look, and for that we will try to remove most of the features that do not corellate at all with the "Showed_up"

In [ ]:
selected_feature = 'Showed_up'

# Find columns with zero correlation to the selected feature
zero_corr_columns = correlation_matrix[
    (abs(correlation_matrix[selected_feature]) < 0.05)
].index.tolist()

# Remove the selected feature itself from the list (if present)
zero_corr_columns = [col for col in zero_corr_columns if col != selected_feature]

# Drop columns with zero correlation
df_filtered = df_merged.drop(columns=zero_corr_columns)

df_copy = df_filtered.copy()
df_copy.drop(columns=["Neighbourhood"], inplace=True)
correlation_matrix = df_copy.corr()
plot_correlation_matrix(correlation_matrix, "Correlation Matrix")

## 4. Prepare the data

For the economic data for the neighbourhoods, there was significant work to be done to translate and prepare it for ingestion. First we used Google Translate to get a rough draft of the translations. This was done by manually extracting the column headers to a Google Sheet and applying a translation operation.

```
=GOOGLETRANSLATE(B2, "pt", "en")
```

These initial machine translations were sent to our Portuguese classmate Laura Do Bem Rebelo for review. It was clear that some of the machine translations needed work.


```
Razão por Sexo -> (Google translate) Reason for sex -> (Laura's edits) Sex ratio
```

From there, the translations with fixes were applied and the data from the 5 tables was set together. Since the neighbourhoods were consistent, it was just a matter of manually combining them.